# Projet 01 - SQL : Interroger une base de réservations de vols

Bienvenue dans le premier projet du cahier de vacances Machine Learnia !

<div align="center">
<img src="./image.png" alt="ouvrir dans VSCode pour voir l'image" width="800"/>
</div>

Tu viens de finir ta dernière journée de travail, et direction l'aéroport de Nice-Côte d'Azur ! Il fait 32 degrés, tu as ta valise à la main et ton vol pour Athènes est dans 3 heures. En attendant, autant pratiquer un peu.

Le but de ce TP est de te glisser dans la peau d'un analyste qui travaille pour l'aéroport. Sous tes yeux, le système de réservations : des passagers, des vols, des réservations. Ta mission sera de poser des questions à ces données et d'en tirer des réponses précises : quels vols décollent de Nice cette semaine ? Quelles destinations rapportent le plus d'argent ? Qui embarque sur le vol de New York ? À la fin du projet, tu sauras transformer n'importe quelle question de ce genre en une requête qui te donne la réponse en quelques millisecondes. Et bonne nouvelle, pas besoin d'être développeur ici pour y arriver : on part de zéro et chaque exercice vérifie tout seul que ta réponse est la bonne.

### L'outil du jour : SQL

Aujourd'hui, nous sommes entourés de bases de données relationnelles : elles font tourner les sites web, les applications mobiles, les banques, les systèmes de gestion et bien sûr les compagnies aériennes. À chaque fois que tu réserves un billet d'avion, une base de données enregistre quelque part qui tu es, quel vol tu prends et quel siège tu occupes.

SQL (Structured Query Language) est le langage standard pour dialoguer avec ces bases : on écrit une question dans un format précis, qu'on appelle une requête, et la base nous renvoie la réponse. Pour un Data Analyst, un Data Engineer ou toute autre personne travaillant avec de la data, c'est sans doute l'outil le plus utilisé au quotidien, et il n'a presque pas changé depuis 50 ans. Un excellent investissement, donc, et c'est pourquoi le premier projet de ce cahier de vacances porte sur SQL !

Ici on utilisera SQLite, une version légère qui tient dans un simple fichier (ou même dans la mémoire de l'ordinateur), parfaite pour apprendre. Rien de très exotique comme technologie, mais les concepts sont exactement les mêmes que sur PostgreSQL ou MySQL, les moteurs qu'on retrouve en entreprise.

Au programme : les requêtes de base, les agrégations, les jointures et les sous-requêtes. Autrement dit, les briques qu'on utilise vraiment tous les jours dans les métiers de la data.

## Imports et configuration

Avant d'écrire la moindre requête, faisons le point sur notre installation. Si tu as suivi le tutoriel du README du repo, tu disposes déjà d'un environnement Python prêt à l'emploi (le dossier `.venv` créé avec `uv`). Un environnement, c'est une bulle isolée qui contient une version de Python et les librairies propres au projet, sans rien toucher au reste de ton ordinateur.

Dans ce projet, on va utiliser deux librairies :

- `sqlite3` : c'est elle qui nous permet de créer notre base de données SQLite et de lui envoyer des requêtes depuis Python. Bonne nouvelle, elle est incluse de base dans Python, il n'y a rien à installer.
- `pandas` : la librairie de référence pour manipuler des données en Python. On s'en servira pour afficher les résultats de nos requêtes sous forme de tableaux bien lisibles (des DataFrames). Elle n'est pas présente de base dans Python, il faut donc l'ajouter à notre environnement.

Pour ajouter `pandas` à l'environnement, ouvre un terminal à la racine du projet et tape :

```shell
uv add pandas
```

> **Important** : pour pouvoir utiliser les notebooks Jupyter il faudra installer la librairie `ipykernel` dans ton environnement. Si tu ne l'as pas encore fait, tape la commande suivante dans le terminal à la racine du projet :
> ```shell
> uv add ipykernel
> ```



Ensuite, une fois dans le notebook, il ne reste qu'à sélectionner le kernel de notre environnement : en haut à droite de l'éditeur, clique sur "Select Kernel" (ou "Sélectionner le noyau") et choisis l'environnement Python du projet, celui qui mentionne `.venv`. C'est lui qui contient le `pandas` qu'on vient d'installer. On peut maintenant lancer nos imports :

In [ ]:
import sqlite3
import pandas as pd

## Création de la base de données

Avant de pouvoir interroger quoi que ce soit, il nous faut... des données ! Dans la vraie vie, la base de réservations existe déjà : c'est le système de la compagnie aérienne qui la remplit à chaque billet vendu. Ici, on va la fabriquer nous-mêmes, en miniature : 15 passagers, 15 vols et une vingtaine de réservations. C'est l'occasion parfaite pour découvrir comment une base de données se construit, avant d'apprendre à l'interroger.

Notre base contient trois tables :

```txt
passengers (id, first_name, last_name, email, nationality)
flights    (id, flight_number, origin, destination, departure_time, arrival_time, aircraft, capacity, price_eur)
bookings   (id, passenger_id, flight_id, booking_date, seat_class, seat_number, status)
```

La table `bookings` fait le lien entre les deux autres : chaque réservation pointe vers un passager (`passenger_id`) et vers un vol (`flight_id`). C'est exactement ça, une base relationnelle : des tables reliées entre elles par des identifiants.

On commence par ouvrir une connexion vers une base SQLite en mémoire. "En mémoire" signifie que la base vit dans la RAM le temps de la session : aucun fichier n'est créé, et la base est recréée à chaque exécution du notebook. C'est voulu, ça évite les surprises si tu relances les cellules dans le désordre.

In [ ]:
conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

print("Connexion à la base établie !")

### Table 1 : les passagers

On crée la première table ensemble, et on en profite pour décortiquer la syntaxe de `CREATE TABLE`. Chaque ligne déclare une colonne avec son nom, son type, et éventuellement une contrainte :

- `id INTEGER PRIMARY KEY` : un entier qui identifie chaque passager de façon unique. C'est la clé primaire, celle vers laquelle les autres tables pourront pointer.
- `first_name TEXT NOT NULL` et `last_name TEXT NOT NULL` : du texte, avec la contrainte `NOT NULL` qui interdit d'enregistrer un passager sans prénom ou sans nom.
- `email TEXT UNIQUE` : la contrainte `UNIQUE` garantit que deux passagers ne peuvent pas partager la même adresse email.
- `nationality TEXT` : du texte, sans contrainte particulière, la colonne peut rester vide.

Une fois la table créée, on la remplit avec `INSERT INTO` via `executemany`, qui insère toutes les lignes d'un coup. Exécute la cellule, elle vérifie toute seule que tout est en place.

In [ ]:
cursor.execute("""
CREATE TABLE passengers (
    id          INTEGER PRIMARY KEY,
    first_name  TEXT NOT NULL,
    last_name   TEXT NOT NULL,
    email       TEXT UNIQUE,
    nationality TEXT
);
""")

passengers_data = [
    (1, 'Lucas',   'Moreau',   'lucas.moreau@email.fr',    'Française'),
    (2, 'Emma',    'Dubois',   'emma.dubois@email.fr',     'Française'),
    (3, 'Noah',    'Leroy',    'noah.leroy@email.fr',      'Française'),
    (4, 'Sofia',   'Rossi',    'sofia.rossi@email.it',     'Italienne'),
    (5, 'Liam',    'Smith',    'liam.smith@email.uk',      'Britannique'),
    (6, 'Mia',     'Müller',   'mia.muller@email.de',      'Allemande'),
    (7, 'Hugo',    'Bernard',  'hugo.bernard@email.fr',    'Française'),
    (8, 'Camille', 'Thomas',   'camille.thomas@email.fr',  'Française'),
    (9, 'Alice',   'Martin',   'alice.martin@email.fr',    'Française'),
    (10,'Carlos',  'Garcia',   'carlos.garcia@email.es',   'Espagnole'),
    (11,'Jade',    'Petit',    'jade.petit@email.fr',      'Française'),
    (12,'Omar',    'Hassan',   'omar.hassan@email.ma',     'Marocaine'),
    (13,'Elena',   'Popescu',  'elena.popescu@email.ro',   'Roumaine'),
    (14,'Antoine', 'Garnier',  'antoine.garnier@email.fr', 'Française'),
    (15,'Yako',    'Tanaka',   'yuki.tanaka@email.jp',     'Japonaise'),
]

cursor.executemany("INSERT INTO passengers VALUES (?,?,?,?,?)", passengers_data)
conn.commit()

cursor.execute("SELECT COUNT(*) FROM passengers")
count = cursor.fetchone()[0]
assert count == 15, f"La table passengers devrait contenir 15 passagers, elle en a {count}"
print(f"Table passengers créée et remplie avec {count} passagers !")

### Table 2 : les vols (à toi de jouer)

Maintenant que tu as vu la syntaxe, c'est toi qui crées la table `flights`. Elle recense les vols avec leurs numéros, leurs horaires, l'avion utilisé et le prix du billet. Complète la commande `CREATE TABLE` entre les balises avec les colonnes suivantes :

| Colonne | Type | Contrainte |
|---|---|---|
| `id` | INTEGER | PRIMARY KEY |
| `flight_number` | TEXT | NOT NULL |
| `origin` | TEXT | NOT NULL |
| `destination` | TEXT | NOT NULL |
| `departure_time` | TEXT | NOT NULL |
| `arrival_time` | TEXT | NOT NULL |
| `aircraft` | TEXT | |
| `capacity` | INTEGER | |
| `price_eur` | REAL | |

N'oublie pas la virgule entre chaque colonne (mais pas après la dernière). Les données sont déjà prêtes dans la cellule, ne les modifie pas : les exercices de la suite du projet comptent dessus, et la cellule vérifie automatiquement ton schéma.

In [ ]:
cursor.execute("""
-- ### START CODE HERE ###

-- ### END CODE HERE ###
""")

flights_data = [
    (1,  'AF1234', 'Nice',      'Paris CDG',    '2026-07-01 07:00', '2026-07-01 08:30', 'Airbus A320', 180, 89.0),
    (2,  'AF1235', 'Paris CDG', 'Nice',          '2026-07-01 19:00', '2026-07-01 20:30', 'Airbus A320', 180, 95.0),
    (3,  'EZ4501', 'Nice',      'Londres',       '2026-07-02 06:30', '2026-07-02 08:15', 'Airbus A319', 156, 120.0),
    (4,  'EZ4502', 'Nice',      'Amsterdam',     '2026-07-02 09:00', '2026-07-02 11:30', 'Airbus A319', 156, 135.0),
    (5,  'VY6010', 'Nice',      'Barcelone',     '2026-07-03 11:00', '2026-07-03 12:45', 'Boeing 737',  189, 78.0),
    (6,  'AF5501', 'Nice',      'New York JFK',  '2026-07-04 13:00', '2026-07-04 22:30', 'Boeing 777',  350, 650.0),
    (7,  'LH2201', 'Nice',      'Francfort',     '2026-07-05 08:00', '2026-07-05 09:45', 'Airbus A320', 180, 112.0),
    (8,  'IB3301', 'Nice',      'Madrid',        '2026-07-05 14:00', '2026-07-05 16:00', 'Boeing 737',  189, 99.0),
    (9,  'AF1236', 'Nice',      'Paris CDG',     '2026-07-06 18:00', '2026-07-06 19:30', 'Airbus A320', 180, 105.0),
    (10, 'U24401', 'Nice',      'Athènes',       '2026-07-07 10:00', '2026-07-07 13:30', 'Boeing 737',  189, 145.0),
    (11, 'FR7701', 'Nice',      'Dublin',        '2026-07-08 07:30', '2026-07-08 10:00', 'Boeing 737',  189, 89.0),
    (12, 'AF1237', 'Paris CDG', 'Nice',          '2026-07-08 20:00', '2026-07-08 21:30', 'Airbus A321', 220, 88.0),
    (13, 'EZ4503', 'Nice',      'Berlin',        '2026-07-09 12:00', '2026-07-09 14:15', 'Airbus A319', 156, 118.0),
    (14, 'AF5502', 'Nice',      'New York JFK',  '2026-07-10 13:00', '2026-07-10 22:30', 'Boeing 777',  350, 680.0),
    (15, 'TK8801', 'Nice',      'Istanbul',      '2026-07-11 09:00', '2026-07-11 12:30', 'Boeing 737',  189, 160.0),
]

cursor.executemany("INSERT INTO flights VALUES (?,?,?,?,?,?,?,?,?)", flights_data)
conn.commit()

cursor.execute("PRAGMA table_info(flights)")
columns = cursor.fetchall()
schema = {col[1]: col[2] for col in columns}
expected_schema = {
    'id': 'INTEGER', 'flight_number': 'TEXT', 'origin': 'TEXT', 'destination': 'TEXT',
    'departure_time': 'TEXT', 'arrival_time': 'TEXT', 'aircraft': 'TEXT',
    'capacity': 'INTEGER', 'price_eur': 'REAL',
}
assert schema == expected_schema, f"Le schéma ne correspond pas. Attendu : {expected_schema}, obtenu : {schema}"

not_null_columns = {col[1] for col in columns if col[3] == 1}
expected_not_null = {'flight_number', 'origin', 'destination', 'departure_time', 'arrival_time'}
assert not_null_columns == expected_not_null, \
    f"Les colonnes NOT NULL attendues sont {expected_not_null}, tu as : {not_null_columns}"

cursor.execute("SELECT COUNT(*) FROM flights")
count = cursor.fetchone()[0]
assert count == 15, f"La table flights devrait contenir 15 vols, elle en a {count}"
print(f"Table flights créée et remplie avec {count} vols, exercice validé !")

### Table 3 : les réservations (à toi de jouer)

Dernière table, et pas des moindres : c'est elle qui relie les passagers aux vols. Deux nouveautés par rapport aux précédentes :

- `REFERENCES` : la colonne `passenger_id` pointe vers l'`id` de la table `passengers`, et `flight_id` vers l'`id` de la table `flights`. C'est ce qu'on appelle une clé étrangère, le ciment des bases relationnelles.
- `CHECK` : cette contrainte restreint les valeurs autorisées dans une colonne. Par exemple `CHECK(seat_class IN ('economy', 'business', 'first'))` refusera toute valeur autre que ces trois classes de siège.

Voici les colonnes à déclarer :

| Colonne | Type | Contrainte |
|---|---|---|
| `id` | INTEGER | PRIMARY KEY |
| `passenger_id` | INTEGER | REFERENCES passengers(id) |
| `flight_id` | INTEGER | REFERENCES flights(id) |
| `booking_date` | TEXT | |
| `seat_class` | TEXT | CHECK parmi `'economy'`, `'business'`, `'first'` |
| `seat_number` | TEXT | |
| `status` | TEXT | CHECK parmi `'confirmed'`, `'cancelled'`, `'pending'` |

Comme avant, ne modifie pas les données : la cellule vérifie le schéma, et teste même que ta contrainte `CHECK` refuse bien une valeur interdite.

In [ ]:
cursor.execute("""
-- ### START CODE HERE ###

-- ### END CODE HERE ###
""")

bookings_data = [
    (1,  1,  1,  '2026-06-01', 'economy',  '14A', 'confirmed'),
    (2,  2,  1,  '2026-06-02', 'business', '2C',  'confirmed'),
    (4,  4,  5,  '2026-06-05', 'economy',  '18C', 'confirmed'),
    (5,  5,  3,  '2026-06-06', 'business', '1A',  'confirmed'),
    (6,  6,  7,  '2026-06-07', 'economy',  '31D', 'confirmed'),
    (7,  7,  6,  '2026-06-08', 'first',    '1A',  'confirmed'),
    (8,  8,  6,  '2026-06-09', 'business', '3B',  'confirmed'),
    (9,  9,  9,  '2026-06-10', 'economy',  '25A', 'confirmed'),
    (10, 10, 5,  '2026-06-11', 'economy',  '17B', 'cancelled'),
    (11, 11, 10, '2026-06-12', 'economy',  '8C',  'confirmed'),
    (12, 12, 15, '2026-06-13', 'economy',  '11A', 'confirmed'),
    (13, 13, 4,  '2026-06-14', 'economy',  '29B', 'pending'),
    (14, 14, 2,  '2026-06-15', 'business', '4A',  'confirmed'),
    (15, 15, 6,  '2026-06-16', 'first',    '2A',  'confirmed'),
    (16, 1,  9,  '2026-06-17', 'economy',  '33C', 'confirmed'),
    (17, 2,  10, '2026-06-18', 'economy',  '15A', 'cancelled'),
    (18, 5,  6,  '2026-06-19', 'business', '5B',  'confirmed'),
    (19, 7,  14, '2026-06-20', 'first',    '1B',  'confirmed'),
    (20, 9,  11, '2026-06-21', 'economy',  '22D', 'confirmed'),
]

cursor.executemany("INSERT INTO bookings VALUES (?,?,?,?,?,?,?)", bookings_data)
conn.commit()

cursor.execute("PRAGMA table_info(bookings)")
schema = {col[1]: col[2] for col in cursor.fetchall()}
expected_schema = {
    'id': 'INTEGER', 'passenger_id': 'INTEGER', 'flight_id': 'INTEGER',
    'booking_date': 'TEXT', 'seat_class': 'TEXT', 'seat_number': 'TEXT', 'status': 'TEXT',
}
assert schema == expected_schema, f"Le schéma ne correspond pas. Attendu : {expected_schema}, obtenu : {schema}"

try:
    cursor.execute("INSERT INTO bookings VALUES (999, 1, 1, '2026-06-30', 'premium', '1A', 'confirmed')")
    raise AssertionError("La contrainte CHECK sur seat_class ne fonctionne pas : 'premium' a été accepté")
except sqlite3.IntegrityError:
    pass

cursor.execute("SELECT COUNT(*) FROM bookings")
count = cursor.fetchone()[0]
assert count == 19, f"La table bookings devrait contenir 19 réservations, elle en a {count}"

print("Base de données complète, exercice validé !")
print(f"  {len(passengers_data)} passagers")
print(f"  {len(flights_data)} vols")
print(f"  {len(bookings_data)} réservations")

## Petite fonction utilitaire

Dernière étape de préparation. Pour interroger la base, on pourrait appeler `pd.read_sql_query(...)` à chaque fois, mais c'est un peu long à écrire. On crée donc une petite fonction `query` qui prend une requête SQL (sous forme de texte) et renvoie directement le résultat sous forme de DataFrame pandas, ce fameux tableau bien lisible. C'est elle que tu utiliseras dans tous les exercices.

La cellule affiche aussi les 5 premières lignes de chaque table, histoire de voir enfin à quoi ressemblent nos données.

In [ ]:
def query(sql):
    """Execute a SQL query and return the result as a pandas DataFrame."""
    return pd.read_sql_query(sql, conn)

print("=== PASSENGERS ===")
display(query("SELECT * FROM passengers LIMIT 5"))
print("=== FLIGHTS ===")
display(query("SELECT * FROM flights LIMIT 5"))
print("=== BOOKINGS ===")
display(query("SELECT * FROM bookings LIMIT 5"))

---

## Partie 1 - Requêtes de base

On commence tranquillement. L'objectif ici c'est de se familiariser avec le schéma et d'écrire des requêtes simples.

### Lister tous les vols au départ de Nice

Récupère le numéro de vol (`flight_number`), la destination, l'heure de départ et le prix, pour tous les vols dont l'origine est `'Nice'`. Trie les résultats par heure de départ.

In [ ]:
result = query("""
-- ### START CODE HERE ###

-- ### END CODE HERE ###
""")
display(result)

assert len(result) == 13, f"Tu devrais avoir 13 vols au départ de Nice, tu en as {len(result)}"
assert list(result.columns) == ['flight_number', 'destination', 'departure_time', 'price_eur'], \
    "Vérifie les colonnes sélectionnées"
print("Exercice validé !")

### Les vols pas chers

Récupère tous les vols dont le prix est inférieur à 100 euros. Affiche le numéro de vol, la destination et le prix. Trie par prix croissant.

> Les low-cost ont encore frappé.

In [ ]:
result = query("""
-- ### START CODE HERE ###

-- ### END CODE HERE ###
""")
display(result)

assert len(result) == 6, f"Tu devrais avoir 6 vols sous 100 euros, tu en as {len(result)}"
assert result['price_eur'].is_monotonic_increasing, "Les résultats doivent être triés par prix croissant"
print("Exercice validé !")

### Recherche par nom de passager

Récupère toutes les infos de la passagère dont le nom de famille est `'Tanaka'`.

In [ ]:
result = query("""
-- ### START CODE HERE ###

-- ### END CODE HERE ###
""")
display(result)

assert len(result) == 1
assert result.iloc[0]['first_name'] == 'Yako'
print("Exercice validé !")

---

## Partie 2 - Agrégations et jointures

On monte d'un cran. `GROUP BY`, `COUNT`, `SUM`, `AVG` et leurs amis (pour extraire des tendances et des statistiques d'une base) ainsi que les jointures (pour croiser des tables).

> Petit rappel rapide sur les types de jointures :
> `INNER JOIN` : seulement les lignes qui ont une correspondance dans les deux tables.
> `LEFT JOIN` : toutes les lignes de la table de gauche, avec les correspondances de droite si elles existent (NULL sinon).

### Nombre de réservations par statut

Compte le nombre de réservations pour chaque statut (`confirmed`, `cancelled`, `pending`). Trie par nombre décroissant.

In [ ]:
result = query("""
-- ### START CODE HERE ###

-- ### END CODE HERE ###
""")
display(result)

assert len(result) == 3, "Il doit y avoir 3 statuts distincts"
confirmed_count = result[result['status'] == 'confirmed']['count'].values[0]
assert confirmed_count == 16, f"Il devrait y avoir 16 réservations confirmées, tu en as {confirmed_count}"
print("Exercice validé !")

### Chiffre d'affaires par destination

Calcule le chiffre d'affaires total (somme des prix) par destination, en ne comptant que les réservations **confirmées**. Trie par CA décroissant.

Pour ça, tu as besoin de joindre `bookings` et `flights`.

**Colonnes attendues :** `destination`, `total_revenue`, `nb_bookings`

In [ ]:
result = query("""
-- ### START CODE HERE ###

-- ### END CODE HERE ###
""")
display(result)

assert 'destination' in result.columns
assert 'total_revenue' in result.columns
assert result['total_revenue'].is_monotonic_decreasing, "Trie par CA décroissant"
top_dest = result.iloc[0]['destination']
assert top_dest == 'New York JFK', f"La destination avec le plus de CA devrait être New York JFK, pas {top_dest}"
print("Exercice validé !")

### Destinations avec plus d'une réservation confirmée

Même idée que l'exercice précédent, mais cette fois on ne garde que les destinations qui ont **strictement plus d'une réservation confirmée**.

> Indice : c'est exactement le moment où `HAVING` devient utile. `WHERE` filtre les lignes avant l'agrégation, `HAVING` filtre après.

In [ ]:
result = query("""
-- ### START CODE HERE ###

-- ### END CODE HERE ###
""")
display(result)

assert all(result['nb_bookings'] > 1), "Toutes les destinations affichées doivent avoir plus d'une réservation"
print("Exercice validé !")

### La liste d'embarquement

Pour le vol `'AF5501'` (Nice, New York), affiche la liste des passagers embarqués avec leur prénom, nom, classe de siège et numéro de siège. Seulement les réservations confirmées.

**Colonnes attendues :** `first_name`, `last_name`, `seat_class`, `seat_number`

In [ ]:
result = query("""
-- ### START CODE HERE ###

-- ### END CODE HERE ###
""")
display(result)

assert len(result) == 4, f"4 passagers confirmés sur ce vol, tu en as {len(result)}"
names = set(result['last_name'].values)
assert names == {'Bernard', 'Thomas', 'Tanaka', 'Smith'}, f"Les passagers attendus sont Garnier, Bernard, Tanaka. Tu as : {names}"
print("Exercice validé !")

### Passagers sans réservation

Trouve les passagers qui n'ont aucune réservation dans la base. Affiche leur prénom et nom.

> Indice : un `LEFT JOIN` combiné avec un filtre sur `NULL` est une façon classique de trouver les "orphelins" dans une table.

In [ ]:
result = query("""
-- ### START CODE HERE ###

-- ### END CODE HERE ###
""")
display(result)

assert len(result) == 1, f"Il y a 1 passagers sans réservation, tu en as {len(result)}"
print("Exercice validé !")

### Itinéraires complets

Affiche pour chaque passager ayant une réservation confirmée : son nom complet (format `Prénom NOM`), le numéro de vol, l'origine, la destination et l'heure de départ. Trie par heure de départ.

**Colonnes attendues :** `full_name`, `flight_number`, `origin`, `destination`, `departure_time`

> Pour concaténer des chaînes en SQL : `first_name || ' ' || last_name`

In [ ]:
result = query("""
-- ### START CODE HERE ###

-- ### END CODE HERE ###
""")
display(result)

assert 'full_name' in result.columns
assert result['departure_time'].is_monotonic_increasing, "Trie par heure de départ"
print("Exercice validé !")

---

## Partie 4 - Sous-requêtes

Les sous-requêtes permettent d'utiliser le résultat d'une requête dans une autre. C'est souvent plus lisible qu'une chaîne de jointures, et ça permet de résoudre des problèmes qu'on ne peut pas traiter avec un simple SELECT.

### Vols plus chers que la moyenne

Récupère tous les vols dont le prix est supérieur au prix moyen de tous les vols. Affiche le numéro de vol, la destination et le prix.

> Indice : tu peux calculer la moyenne dans une sous-requête dans le WHERE.

In [ ]:
result = query("""
-- ### START CODE HERE ###

-- ### END CODE HERE ###
""")
display(result)

avg_price = pd.read_sql_query("SELECT AVG(price_eur) as avg FROM flights", conn)['avg'].values[0]
assert all(result['price_eur'] > avg_price), f"Tous les vols doivent être au-dessus de la moyenne ({avg_price:.2f} euros)"
print(f"Prix moyen : {avg_price:.2f} euros")
print("Exercice validé !")

### Les grands voyageurs

Trouve les passagers qui ont au moins 2 réservations (tous statuts confondus). Affiche leur prénom, nom et le nombre de réservations.

> Deux façons de faire ça : avec une sous-requête, ou avec un GROUP BY + HAVING. Choisis celle que tu préfères.

In [ ]:
result = query("""
-- ### START CODE HERE ###

-- ### END CODE HERE ###
""")
display(result)

assert len(result) == 5, f"Il y a 5 passagers avec 2+ réservations, tu en as {len(result)}"
assert all(result['nb_bookings'] >= 2)
print("Exercice validé !")

---

## Bilan

Si tu as validé tous ces exercices, tu maîtrises les bases du SQL qu'on utilise au quotidien en data science. Dans un projet réel, la plupart des requêtes d'exploration et d'analyse s'écrivent avec exactement ces briques.

Bonnes vacances et à bientôt pour le prochain projet !